In [7]:
#imports
import pandas as pd
import string
import re
import warnings

warnings.filterwarnings('ignore')

In [6]:
#TRAIN 
train = pd.read_csv('train.csv')
print(f'{train.head(5)}\n')
print(f'{train.describe()}\n')
print(f'{train.info()}\n')
print(f'{train.isnull().sum()}\n')


   ID                                               text               label
0   1  Atât e de înfricoșat de gândești că e turbat a...  graiul moldovenesc
1   3  Cei ce n-au dreptate n-ar mai năzui în veci la...  graiul moldovenesc
2   5  Deoarece ați primit bani de la oaspetele dumne...  graiul moldovenesc
3   7  Primiți vă rog oameni buni această mică mulțăm...  graiul moldovenesc
4   9  Fiin'c-o vint la noi vezi binie baș în zâua dă...     graiul bănățean

               ID
count  424.000000
mean   424.000000
std    245.085019
min      1.000000
25%    212.500000
50%    424.000000
75%    635.500000
max    847.000000

<class 'pandas.DataFrame'>
RangeIndex: 424 entries, 0 to 423
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   ID      424 non-null    int64
 1   text    424 non-null    str  
 2   label   424 non-null    str  
dtypes: int64(1), str(2)
memory usage: 10.1 KB
None

ID       0
text     0
label    0
dtype: int64



In [3]:
#TEST
test = pd.read_csv('test.csv')


In [26]:
#SUBTASK 1
pattern_pani = r'\bpâni\b'
sumTrain = train['text'].str.count(pattern_pani, flags=re.IGNORECASE).sum()
sumTest = test['text'].str.count(pattern_pani, flags=re.IGNORECASE).sum()
answer = int(sumTrain + sumTest)

task1 = pd.DataFrame({
    'subtaskID': [1],
    'datapointID': [1],
    'answer': [answer]
})

In [23]:
#SUBTASK 2
semne = list(string.punctuation)
semne = [re.escape(s) for s in semne]

semneMLD = (
    train[train['label'] == 'graiul moldovenesc']['text']
    .str.count("|".join(semne), flags=re.IGNORECASE)
    .mean()
)

semneBNT = (
    train[train['label'] == 'graiul bănățean']['text']
    .str.count("|".join(semne), flags=re.IGNORECASE)
    .mean()
)

answer = abs(semneMLD - semneBNT)
answer = round(answer, 2)

task2 = pd.DataFrame({
    'subtaskID': [2],
    'datapointID': [1],
    'answer': [answer]
})

In [20]:
#SUBTASK 3 
diacritice = ['ă', 'â', 'î', 'ș', 'ț', 'Ă', 'Â', 'Î', 'Ș', 'Ț' ]
pattern = "|".join([re.escape(d) for d in diacritice])

answer = test['text'].str.count(pattern)

task3 = pd.DataFrame({
    'subtaskID': 3,
    'datapointID': test['ID'],
    'answer':answer
})

In [33]:
#SUBTASK 4
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import VotingClassifier

union = FeatureUnion([
    ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 3), sublinear_tf=True)),
    ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 6), sublinear_tf=True))
])
label = LabelEncoder()

X = train['text']
y = train['label']

X = union.fit_transform(X)
y = label.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [34]:
model = VotingClassifier(
    estimators=[
        ('mnb', MultinomialNB(alpha=0.05)),
        ('svc', LinearSVC(C=1.0, dual='auto', random_state=42)),
        ('lr', LogisticRegression(C=10.0, max_iter=1000))
    ],
    voting='hard'
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f'F1 Score: {f1_score(y_test, y_pred, average='weighted')} \n')

F1 Score: 0.9764672927994728 



In [37]:
X_sub = test['text']
X_sub = union.transform(X_sub)

y_sub_num = model.predict(X_sub)
y_sub_text = label.inverse_transform(y_sub_num)

answer = y_sub_text

task4 = pd.DataFrame({
    'subtaskID': 4,
    'datapointID': test['ID'],
    'answer': answer
})

In [38]:
#SUBMISSION

submission = pd.concat((task1,task2,task3,task4))
submission.columns = ['subtaskID','datapointID','answer']
submission.to_csv('submission.csv',index=False)